# TP 3 — Contours et reconnaissance de motifs

**Objectifs**

- Calculer un gradient et une magnitude de Sobel à la main avec NumPy.
- Comparer Sobel et Canny pour la détection de bords.
- Détecter des lignes droites avec la transformation de Hough probabiliste.
- Localiser un motif dans une image complexe via *template matching*.

**Durée indicative : 50 minutes.**

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import patches
from scipy import ndimage as ndi
from skimage import color, data, feature, filters, transform, util

img = data.camera()  # niveaux de gris uint8
img_f = util.img_as_float(img)

## Exercice 1 — Sobel à la main

1. Définissez les noyaux de Sobel `kx` et `ky` (matrices 3×3).
2. Convoluez l'image en niveaux de gris avec chacun de ces noyaux à l'aide de `scipy.ndimage.convolve`.
3. Calculez la **magnitude** du gradient : `mag = sqrt(gx² + gy²)`.
4. Affichez `gx`, `gy` et `mag` côte à côte (utilisez `cmap="gray"`).

<details>
<summary>💡 Coup de pouce — calculer Sobel à la main</summary>

**🎯 But :** comprendre que Sobel = juste deux convolutions avec des noyaux fixes. Faire l'implémentation à la main pour démystifier.

**Les noyaux Sobel**

```python
kx = np.array([[-1, 0, 1],
               [-2, 0, 2],
               [-1, 0, 1]])     # détecte les bords verticaux (gradient horizontal)

ky = kx.T                        # transposée → détecte bords horizontaux
```

- La somme de chaque noyau = 0 → un fond uniforme donne 0 (pas de gradient = pas de bord).
- Les coefficients ±1 et ±2 viennent d'une combinaison de **lissage gaussien** + **différenciation finie**.

**Appliquer la convolution**

```python
from scipy.ndimage import convolve
img_f = img.astype(float)           # IMPORTANT : float, sinon overflow
gx = convolve(img_f, kx)
gy = convolve(img_f, ky)
```

⚠️ **Pourquoi convertir en float ?** Sur uint8, un produit `-1 × 255 = -255` déborde et donne `1` (modulo 256). On perd tout le signe. Toujours caster en float avant convolution.

**Calculer la magnitude**

```python
mag = np.sqrt(gx ** 2 + gy ** 2)
```

- `gx`, `gy` sont **signés** (positif ou négatif selon le sens du gradient).
- La magnitude `sqrt(gx² + gy²)` est **toujours positive** → un seul nombre par pixel qui mesure « combien il y a de gradient à cet endroit ».
- Pour afficher proprement, on normalise : `mag = (mag / mag.max() * 255).astype(np.uint8)`.

**Afficher `gx` et `gy` séparément**

Comme `gx` et `gy` sont **signés**, utilisez une colormap divergente type `RdBu` ou `coolwarm` pour bien voir les deux sens :

```python
ax.imshow(gx, cmap="RdBu", vmin=-200, vmax=200)
```

→ rouge = transition sombre → clair, bleu = transition claire → sombre.

</details>

In [2]:
# TODO

## Exercice 2 — Sobel vs Canny

1. Calculez la magnitude de Sobel via `skimage.filters.sobel`.
2. Calculez les contours de Canny via `skimage.feature.canny` avec `sigma=2`, `low_threshold=0.1`, `high_threshold=0.2`.
3. Affichez l'originale, le Sobel, et le Canny. Commentez : pourquoi le Canny donne-t-il des contours fins là où le Sobel donne une carte dense ?

<details>
<summary>💡 Coup de pouce — Sobel skimage vs Canny</summary>

**🎯 But :** voir la différence visuelle entre la magnitude Sobel brute et le résultat propre de Canny.

**Sobel via skimage**

```python
from skimage import filters
sobel_mag = filters.sobel(img)
```

Cette fonction fait pour vous : calcul de `gx` et `gy` + magnitude `sqrt(gx² + gy²)`. Pas besoin de réimplémenter comme à l'Exercice 1.

⚠️ **Sortie en float [0, 1]**, normalisée. Plus besoin de gérer le `.max()` à la main.

**Canny via skimage**

```python
from skimage import feature, util
img_float = util.img_as_float(img)            # conversion uint8 → float si nécessaire
edges = feature.canny(img_float, sigma=2.0, low_threshold=0.1, high_threshold=0.2)
```

⚠️ **Canny attend du float**. Si vous partez d'une image uint8, conversion obligatoire avec `util.img_as_float`.

**Les 3 paramètres clés**
- `sigma` : largeur du flou gaussien initial. Petit = sensible au bruit, grand = manque les détails fins.
- `low_threshold` : seuil bas de l'hystérésis (en fraction du gradient max).
- `high_threshold` : seuil haut. Un pixel > `high_threshold` est sûr ; entre les deux, gardé seulement s'il est connecté à un pixel sûr ; en-dessous de `low_threshold`, jeté.

**Différence visuelle attendue**

| | Sobel magnitude | Canny |
|---|---|---|
| Type de sortie | Float continu | Booléen 0/1 |
| Épaisseur des bords | 2-5 px | **1 px exactement** |
| Bruit | Présent | Filtré par hystérésis |
| Lisibilité | Brute | Propre |

C'est pour ça qu'on enchaîne souvent Canny avec Hough : on a besoin de bords **fins et binaires** pour que Hough vote correctement.

</details>

In [3]:
# TODO

## Exercice 3 — Détection de lignes par Hough

On utilise une image avec des lignes claires (`data.checkerboard` ou une image synthétique).

1. Construisez une image avec quelques lignes (utilisez `np.zeros` puis tracez 3-4 lignes via `skimage.draw.line`).
2. Ajoutez un peu de bruit gaussien.
3. Calculez les contours avec Canny.
4. Appliquez `transform.probabilistic_hough_line` pour récupérer les segments détectés.
5. Tracez l'image originale et superposez les segments détectés (en rouge).

<details>
<summary>💡 Coup de pouce — détection de lignes par Hough probabiliste</summary>

**🎯 But :** détecter automatiquement les segments de droite dans une image (marquage au sol, bords de table, etc.).

**Synthétiser une image de test**

```python
from skimage.draw import line
img = np.zeros((200, 300), dtype=np.uint8)
rr, cc = line(20, 30, 180, 270)      # ligne de (y0=20, x0=30) à (y1=180, x1=270)
img[rr, cc] = 255
```

`skimage.draw.line` retourne les **coordonnées des pixels** sur la ligne. `img[rr, cc] = 255` les allume tous.

**Ajouter du bruit (pour rendre l'exo réaliste)**

```python
noisy = img + np.random.normal(0, 5, img.shape)
noisy = np.clip(noisy, 0, 255).astype(np.uint8)
```

**Détecter avec Hough probabiliste**

```python
from skimage.transform import probabilistic_hough_line
edges = feature.canny(noisy / 255, sigma=1.5)
segments = probabilistic_hough_line(edges, threshold=10, line_length=30, line_gap=3)
```

Les 3 paramètres :
- `threshold` : nombre minimum de votes pour qu'une droite soit considérée.
- `line_length` : longueur minimale d'un segment (en pixels) pour être renvoyé. Filtre les petits détails.
- `line_gap` : si deux pixels alignés sont à ≤ ce nombre de pixels l'un de l'autre, ils appartiennent au même segment (gère les petites cassures).

**Format de sortie**

```python
segments = [((x0, y0), (x1, y1)), ((x0, y0), (x1, y1)), ...]
```

Une **liste de tuples**. Chaque tuple = 2 extrémités d'un segment en coordonnées **(x, y)** (donc en convention matplotlib, pas NumPy).

**Tracer**

```python
for (x0, y0), (x1, y1) in segments:
    ax.plot([x0, x1], [y0, y1], 'r-', linewidth=2)
```

Note : `ax.plot` attend `([x_vals], [y_vals])` séparés, d'où le `[x0, x1], [y0, y1]`.

</details>

In [4]:
# TODO — créer image avec lignes et la traiter

## Exercice 4 — Template matching

1. Récupérez l'image `data.coins()` et extrayez la pièce située dans la zone `[140:200, 220:280]`. Cette extraction sert de **template**.
2. Avec `feature.match_template`, calculez la carte de corrélation entre `coins` et le template.
3. Trouvez la position du **meilleur match** via `np.unravel_index`.
4. Affichez l'image, le template, la carte de corrélation, et tracez un rectangle (avec `matplotlib.patches.Rectangle`) autour de la position détectée.

**Bonus** : trouvez les 3 meilleures positions (locales) en seuillant la carte de corrélation. Vous pouvez utiliser `feature.peak_local_max`.

<details>
<summary>💡 Coup de pouce — template matching</summary>

**🎯 But :** trouver toutes les positions où un petit motif (« template ») apparaît dans une grande image, par corrélation glissante.

**Extraire le template**

```python
template = img[140:200, 220:280]   # par ex. une pièce isolée à ces coords
```

C'est juste un slicing NumPy. Le template doit être **plus petit** que l'image dans laquelle on cherche.

**Calculer la carte de corrélation**

```python
from skimage.feature import match_template
result = match_template(img, template)
```

`result` est une **carte 2D** de score de corrélation [-1, 1] :
- 1 = match parfait (le template apparaît exactement à cet endroit)
- ≈ 0 = pas de corrélation
- -1 = anti-corrélation (image inversée)

⚠️ **Taille de `result`** : `result.shape = (H_img - H_template + 1, W_img - W_template + 1)`. Plus petite que l'image originale (on ne peut pas tester près des bords).

**Trouver le pic maximum**

```python
y, x = np.unravel_index(np.argmax(result), result.shape)
```

- `np.argmax(result)` renvoie l'index **plat** (1D) du max.
- `np.unravel_index(idx, shape)` le convertit en coordonnées 2D `(y, x)`.

Cette `(y, x)` est le **coin haut-gauche** du match, pas son centre.

**Tracer un rectangle autour du match**

```python
from matplotlib.patches import Rectangle
ax.add_patch(Rectangle((x, y), template.shape[1], template.shape[0],
                       fill=False, edgecolor='red', linewidth=2))
```

`Rectangle` veut `(x, y)` puis `(w, h)`. Et `template.shape = (h, w)` donc on prend `template.shape[1]` pour la largeur, `[0]` pour la hauteur.

**Bonus — détecter TOUS les matches (pas juste le meilleur)**

```python
from skimage.feature import peak_local_max
peaks = peak_local_max(result, min_distance=20, threshold_rel=0.7)
# peaks = array de [(y, x), (y, x), ...]
```

- `min_distance` : pics séparés d'au moins X px (non-max suppression).
- `threshold_rel=0.7` : ne garde que les pics dont le score est ≥ 70 % du max global.

</details>

In [5]:
# TODO